In [6]:
!pip uninstall -y langchain langchain-classic langchain-core langchain-community langchain-text-splitters langchain-experimental langchain-protocol

Found existing installation: langchain 1.2.16
Uninstalling langchain-1.2.16:
  Successfully uninstalled langchain-1.2.16
Found existing installation: langchain-classic 1.0.4
Uninstalling langchain-classic-1.0.4:
  Successfully uninstalled langchain-classic-1.0.4
Found existing installation: langchain-core 1.3.2
Uninstalling langchain-core-1.3.2:
  Successfully uninstalled langchain-core-1.3.2
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langchain-experimental 0.4.1
Uninstalling langchain-experimental-0.4.1:
  Successfully uninstalled langchain-experimental-0.4.1
Found existing installation: langchain-protocol 0.0.14
Uninstalling langchain-protocol-0.0.14:
  Successfully uninstalled langchain-prot

In [7]:
!pip install "langchain==0.3.27" "langchain-community==0.3.27" "langchain-text-splitters==0.3.11" "langchain-experimental==0.3.4" "langchain-core==0.3.78" sentence-transformers faiss-cpu

  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
  Using cached langchain_experimental-0.3.4-py3-none-any.whl.metadata (1.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 65.1 MB/s eta 0:00:00
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
Using cached langchain_experimental-0.3.4-py3-none-any.whl (209 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 34.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.10 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.78 which is incompatible.
langgraph-prebuilt 1.0.12 requires langchain-core>=1.3.1, but you have langchain-core 0.3.78 which is incompatible.


In [1]:
!pip show langchain | grep Version

Version: 0.3.27


In [3]:
# fix_4_context_compression.py
import pandas as pd
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_experimental.text_splitter import SemanticChunker

In [4]:
# Load and chunk the dataset
df = pd.read_csv("/content/customer_support_tickets.csv")
documents = df["Ticket Description"].dropna().tolist()

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
splitter   = SemanticChunker(embeddings=embeddings, breakpoint_threshold_type="percentile")

chunks = []
for doc in documents[:300]:   # use first 300 tickets for demo
    chunks.extend(splitter.split_text(doc))

print(f"Total semantic chunks indexed: {len(chunks)}")


/tmp/ipykernel_7652/3769228324.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warn

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total semantic chunks indexed: 600


In [5]:
# Build vector store
vectorstore    = FAISS.from_texts(chunks, embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})


In [6]:
# Add compression filter
# EmbeddingsFilter removes chunks whose similarity to the query falls below threshold
compressor = EmbeddingsFilter(
    embeddings=embeddings,
    similarity_threshold=0.76  # only keep chunks above this relevance bar
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [7]:
# Compare retrieval: uncompressed vs. compressed
query = "What is the refund policy for billing errors?"

raw_docs        = base_retriever.get_relevant_documents(query)
compressed_docs = compression_retriever.get_relevant_documents(query)

print(f"\nQuery: '{query}'")
print(f"\nUncompressed retrieval : {len(raw_docs)} chunks")
print(f"Compressed retrieval   : {len(compressed_docs)} chunks")

print("\n── Compressed context passed to LLM ──")
for i, doc in enumerate(compressed_docs):
    print(f"\n  Chunk {i+1}:\n  {doc.page_content[:250]}...")

/tmp/ipykernel_7652/2628348211.py:4: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  raw_docs        = base_retriever.get_relevant_documents(query)



Query: 'What is the refund policy for billing errors?'

Uncompressed retrieval : 10 chunks
Compressed retrieval   : 0 chunks

── Compressed context passed to LLM ──
